# HDB resale data engineering assessment

## Goal
Prepare the supplied resale snapshots for January 2012 to December 2016, with a
traceable decision for every row. The Python package performs the ETL; this
notebook runs it and presents the evidence. **No data are fetched from a URL.**

The submitted execution produces **83,612 cleaned records** and **8,932
quarantined records** from **92,544** scoped rows. The original five CSV files
contain **986,276** rows; other periods remain unchanged in Raw.

The two AWS architecture designs are in `architecture/`; detailed decisions and
sources are in `docs/architecture.md`. The assessment brief itself is excluded.


## Setup
From the repository root, install the pinned environment with
`python -m pip install -r requirements-lock.txt` followed by
`python -m pip install -e . --no-deps`.

The notebook can be executed with `python scripts/execute_notebook.py`.
It does not download data, install packages from a cell, or require AWS credentials.


In [1]:
from pathlib import Path
import importlib.metadata
import json
import sys
import pandas as pd
from IPython.display import display

# Resolve the project from this notebook's working directory.
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Start the notebook from the repository root')
sys.path.insert(0, str(ROOT / 'src'))
from hdb_resale.pipeline import Config, run_pipeline, sha256_file

INPUT_DIR = ROOT / 'data' / 'raw'
OUTPUT_DIR = ROOT / 'data' / 'outputs'
CONFIG = Config(start_month='2012-01', end_month='2016-12', reference_month='2012-01')
display(pd.DataFrame({'package': ['pandas', 'numpy', 'pyarrow'],
                      'version': [importlib.metadata.version(p) for p in ['pandas', 'numpy', 'pyarrow']]}))


,package,version
0,pandas,2.2.3
1,numpy,2.3.5
2,pyarrow,25.0.1


## Key assumptions

- Date inherits the January 2012 **format** (`YYYY-MM`), while its values may span
  the requested 60 months. Town, flat type, flat model and storey range must belong
  to the normalized January 2012 domains. Unknown categories are quarantined.
- Unicode, case and whitespace normalization correct representation only. Five-storey
  bands are not converted into three-storey bands because the exact floor is unknown.
- Remaining lease is measured **at the transaction month**, assuming a 99-year
  term starting on 1 January of the supplied commencement year. Months use floor
  division; the source `remaining_lease` is preserved separately. This is an estimate.
- The composite key contains every source attribute except `resale_price`, including
  the union-schema `remaining_lease`. Lineage and derived attributes are excluded.
- Validation runs before maximum-price deduplication; potential anomalies are then
  quarantined without promoting a lower-price duplicate. The group mean for the
  identifier is calculated on the final cleaned records.
- A nine-character business code is not a unique transaction key. The literal
  SHA-256 code hash is supplied along with a unique SHA-256 composite-key hash.


## Steps
### 1. Execute the pipeline
Read the CSVs as-is in chunks; union their attributes; retain the requested period;
profile and validate; select the maximum-price record per key; flag price anomalies;
generate identifiers and hashes; write the four derived output groups and reports.

Raw is `data/raw/`. Well-formed out-of-period rows are scope exclusions, not bad
records. A malformed date would be routed into the master for quarantine.


In [2]:
# Record raw checksums before execution; the pipeline never writes to INPUT_DIR.
raw_before = {p.name: sha256_file(p) for p in sorted(INPUT_DIR.glob('*.csv'))}
summary = run_pipeline(INPUT_DIR, OUTPUT_DIR, CONFIG)
counts = summary['counts']
display(pd.DataFrame(counts.items(), columns=['metric', 'rows_or_count']))


,metric,rows_or_count
0,raw_rows,986276
1,out_of_scope_rows,893732
2,in_scope_rows,92544
3,invalid_date_rows,0
4,master_rows,92544
5,reference_rows,1559
6,validation_rejected_rows,7411
7,validation_passed_rows,85133
8,duplicate_rejected_rows,1383
9,anomaly_candidates,138


### 2. Inspect source coverage and the union schema
All five inputs are retained. The two files outside the requested period are still
scanned for their schemas and included in the raw inventory. An absent attribute
becomes null in the combined master; no source column is discarded.


In [3]:
inventory = json.loads((OUTPUT_DIR / 'reports/source_inventory.json').read_text())
display(pd.DataFrame(inventory)[['file', 'rows', 'in_scope_rows', 'out_of_scope_rows', 'min_month', 'max_month']])
print('Union attributes:', ', '.join(summary['source_columns']))
print('Composite-key attributes:', ', '.join(summary['composite_key_columns']))


,file,rows,in_scope_rows,out_of_scope_rows,min_month,max_month
0,"Resale Flat Prices (Based on Approval Date), 1...",287196,0,287196,1990-01,1999-12
1,"Resale Flat Prices (Based on Approval Date), 2...",369651,3188,366463,2000-01,2012-02
2,Resale Flat Prices (Based on Registration Date...,37153,37153,0,2015-01,2016-12
3,Resale Flat Prices (Based on Registration Date...,52203,52203,0,2012-03,2014-12
4,Resale flat prices based on registration date ...,240073,0,240073,2017-01,2026-09


Union attributes: block, flat_model, flat_type, floor_area_sqm, lease_commence_date, month, remaining_lease, resale_price, storey_range, street_name, town
Composite-key attributes: block, flat_model, flat_type, floor_area_sqm, lease_commence_date, month, remaining_lease, storey_range, street_name, town


### 3. Review the authoritative domains and profiling
The reference is generated from the 1,559 January 2012 records. Later additions are
not silently folded into the reference. The complete profiles include missingness,
distinct values, top values and numeric distributions, before and after cleaning.


In [4]:
reference = json.loads((OUTPUT_DIR / 'reports/reference_domains.json').read_text())
display(pd.DataFrame([{'attribute': k, 'distinct_reference_values': len(v), 'allowed_values': ', '.join(v)}
                      for k, v in reference['domains'].items()]))
profiles = json.loads((OUTPUT_DIR / 'reports/data_profiles.json').read_text())
display(pd.DataFrame(profiles['master']['columns'])[['column', 'missing', 'missing_pct', 'distinct_non_null']])


,attribute,distinct_reference_values,allowed_values
0,town,26,"ANG MO KIO, BEDOK, BISHAN, BUKIT BATOK, BUKIT ..."
1,flat_type,7,"1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUT..."
2,flat_model,13,"ADJOINED FLAT, APARTMENT, IMPROVED, MAISONETTE..."
3,storey_range,12,"01 TO 03, 04 TO 06, 07 TO 09, 10 TO 12, 13 TO ..."


,column,missing,missing_pct,distinct_non_null
0,block,0,0.0000,2139
1,flat_model,0,0.0000,20
2,flat_type,0,0.0000,7
3,floor_area_sqm,0,0.0000,168
4,lease_commence_date,0,0.0000,48
5,month,0,0.0000,60
6,remaining_lease,55391,59.8537,50
7,resale_price,0,0.0000,2615
8,storey_range,0,0.0000,25
9,street_name,0,0.0000,522


### 4. Review quarantined records
Rows have one terminal quarantine stage and can have multiple validation reasons.
The 492 flat-model failures and 6,975 storey-range failures overlap, so their sum
does not equal the 7,411 validation-rejected rows. These can be legitimate source
changes; they are reference-policy failures, not evidence of corrupt transactions.


In [5]:
cleaned = pd.read_parquet(OUTPUT_DIR / 'cleaned/cleaned.parquet')
transformed = pd.read_parquet(OUTPUT_DIR / 'transformed/transformed.parquet')
quarantined = pd.read_parquet(OUTPUT_DIR / 'quarantined/quarantined.parquet')
hashed = pd.read_parquet(OUTPUT_DIR / 'hashed/hashed.parquet')
display(pd.read_csv(OUTPUT_DIR / 'reports/quarantine_by_reason.csv'))
display(pd.read_csv(OUTPUT_DIR / 'reports/unseen_categories.csv'))
display(quarantined[['month', 'town', 'flat_type', 'resale_price', '_quarantine_stage', '_quarantine_reasons']].head(8))


,reason,rows
0,duplicate_non_maximum_or_tied_price,1383
1,not_in_reference:flat_model,492
2,not_in_reference:storey_range,6975
3,potential_price_anomaly,138


,column,value,rows
0,flat_model,DBSS,277
1,flat_model,TYPE S1,138
2,flat_model,TYPE S2,55
3,flat_model,IMPROVED-MAISONETTE,10
4,flat_model,PREMIUM MAISONETTE,6
5,flat_model,PREMIUM APARTMENT LOFT,5
6,flat_model,2-ROOM,1
7,storey_range,01 TO 05,2700
8,storey_range,06 TO 10,2474
9,storey_range,11 TO 15,1259


,month,town,flat_type,resale_price,_quarantine_stage,_quarantine_reasons
0,2012-01,ANG MO KIO,3 ROOM,336000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"
1,2012-01,ANG MO KIO,3 ROOM,340000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"
2,2012-01,ANG MO KIO,3 ROOM,350000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"
3,2012-01,ANG MO KIO,3 ROOM,334000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"
4,2012-01,BEDOK,3 ROOM,316000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"
5,2012-01,BUKIT MERAH,3 ROOM,303000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"
6,2012-01,BUKIT MERAH,3 ROOM,290000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"
7,2012-01,CHOA CHU KANG,4 ROOM,385000,duplicate,"[""duplicate_non_maximum_or_tied_price""]"


### 5. Check the remaining lease calculation
For a transaction in February 2015 with a commencement year of 1980:

$$R_m = 12(1980 + 99 - 2015) - (2-1) = 767\text{ months}$$
$$R_y=\lfloor 767/12\rfloor=63,\qquad R_{m,\mathrm{remainder}}=767\bmod12=11.$$

The result is 63 years 11 months. The start-day/month is not supplied, so exact legal
lease expiry cannot be inferred. Supplied remaining-lease values are retained; a
difference beyond 12 months is a review warning, not an automatic correction.


In [6]:
lease_columns = ['month', 'town', 'block', 'lease_commence_date', 'remaining_lease',
                 'remaining_lease_years_months', '_provided_lease_difference_months']
display(cleaned.loc[cleaned.month.ge('2015-01'), lease_columns].head(6))
display(cleaned.loc[cleaned._provided_lease_difference_months.abs().gt(12), lease_columns])


,month,town,block,lease_commence_date,remaining_lease,remaining_lease_years_months,_provided_lease_difference_months
47441,2015-01,ANG MO KIO,174,1986,70,70 years 00 months,0.0
47442,2015-01,ANG MO KIO,541,1981,65,65 years 00 months,0.0
47443,2015-01,ANG MO KIO,163,1980,64,64 years 00 months,0.0
47444,2015-01,ANG MO KIO,446,1979,63,63 years 00 months,0.0
47445,2015-01,ANG MO KIO,557,1980,64,64 years 00 months,0.0
47446,2015-01,ANG MO KIO,603,1980,64,64 years 00 months,0.0


,month,town,block,lease_commence_date,remaining_lease,remaining_lease_years_months,_provided_lease_difference_months
66720,2016-02,PASIR RIS,513,1993,77,75 years 11 months,13.0


### 6. Review price anomaly candidates
For each deduplicated valid row, score $x=\log(\mathrm{resale\ price}/\mathrm{floor\ area})$.
Use outer Tukey fences $[Q_1-3\,IQR,\;Q_3+3\,IQR]$ within town / flat type / year,
requiring at least 20 peers and nonzero spread. Fall back to town / flat type over
the full period, then flat type / year. If no peer group qualifies, mark the row
unassessed instead of fabricating a score.

138 records are flagged for review, including legitimate premium or unusual units.
21 retained records have no usable peer group. This retrospective heuristic does
not constitute a predictive model, a fraud label or a validated valuation.


In [7]:
anomalies = quarantined.loc[quarantined._quarantine_stage.eq('anomaly')]
display(anomalies[['month', 'town', 'flat_type', 'resale_price', 'floor_area_sqm', 'price_per_sqm',
                   '_anomaly_peer_level', '_anomaly_peer_count', '_anomaly_lower_ppsqm', '_anomaly_upper_ppsqm']].head(8))
display(cleaned._anomaly_peer_level.value_counts().rename_axis('peer_level').to_frame('rows'))


,month,town,flat_type,resale_price,floor_area_sqm,price_per_sqm,_anomaly_peer_level,_anomaly_peer_count,_anomaly_lower_ppsqm,_anomaly_upper_ppsqm
13,2012-01,KALLANG/WHAMPOA,3 ROOM,705000,79,8924.050633,town_flat_type_year,244.0,3638.255252,8468.249584
19,2012-02,BEDOK,4 ROOM,598000,85,7035.294118,town_flat_type_year,333.0,3349.896900,6880.823042
6898,2012-06,KALLANG/WHAMPOA,3 ROOM,825000,92,8967.391304,town_flat_type_year,244.0,3638.255252,8468.249584
6942,2012-07,KALLANG/WHAMPOA,3 ROOM,810000,94,8617.021277,town_flat_type_year,244.0,3638.255252,8468.249584
7019,2012-09,ANG MO KIO,4 ROOM,680000,91,7472.527473,town_flat_type_year,203.0,3601.757709,7273.358888
7021,2012-09,BEDOK,4 ROOM,698000,97,7195.876289,town_flat_type_year,333.0,3349.896900,6880.823042
7033,2012-09,KALLANG/WHAMPOA,3 ROOM,780000,83,9397.590361,town_flat_type_year,244.0,3638.255252,8468.249584
7087,2012-10,YISHUN,4 ROOM,247000,104,2375.000000,town_flat_type_year,506.0,2804.837824,6510.655352


,rows
peer_level,
town_flat_type_year,82581
town_flat_type_all_years,918
flat_type_year,92
unassessed,21


### 7. Inspect identifiers and hashes
The format is `S` + three block digits + two leading group-average-price digits +
two month digits + town initial. For block 19, group mean $230,000, January and
Ang Mo Kio, the result is `S0192301A`.

The cleaned data contain 71,678 distinct codes across 83,612 rows. A SHA-256 hash
of an identical code is identical, so a hash alone cannot repair this ambiguity.
`resale_identifier_hash` follows the literal hashing instruction; `resale_record_hash`
hashes the unambiguous canonical composite key and is unique in this output.
The pipeline checks for a hash collision and stops if one is detected.


In [8]:
display(transformed[['month', 'town', 'flat_type', 'block', 'resale_price',
                     'group_average_resale_price', 'Resale Identifier']].head(8))
display(hashed[['month', 'town', 'block', 'resale_identifier_hash', 'resale_record_hash']].head(3))
collisions = transformed.loc[transformed['Resale Identifier'].duplicated(keep=False)]
example_code = collisions['Resale Identifier'].iloc[0]
display(collisions.loc[collisions['Resale Identifier'].eq(example_code),
                       ['month', 'town', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'Resale Identifier']])


,month,town,flat_type,block,resale_price,group_average_resale_price,Resale Identifier
0,2012-01,ANG MO KIO,2 ROOM,406,257800.0,256966.666667,S4062501A
1,2012-01,ANG MO KIO,2 ROOM,314,263000.0,256966.666667,S3142501A
2,2012-01,ANG MO KIO,2 ROOM,314,275000.0,256966.666667,S3142501A
3,2012-01,ANG MO KIO,2 ROOM,170,260000.0,256966.666667,S1702501A
4,2012-01,ANG MO KIO,2 ROOM,174,226000.0,256966.666667,S1742501A
5,2012-01,ANG MO KIO,2 ROOM,508,260000.0,256966.666667,S5082501A
6,2012-01,ANG MO KIO,3 ROOM,174,281000.0,348328.654737,S1743401A
7,2012-01,ANG MO KIO,3 ROOM,216,375000.0,348328.654737,S2163401A


,month,town,block,resale_identifier_hash,resale_record_hash
0,2012-01,ANG MO KIO,406,e22d10c12612bb3c3d1ff735a675db47039477c6ce8dbe...,84d847c960678c0e2e895b1416214e54f7a6976d372805...
1,2012-01,ANG MO KIO,314,df36be696d8b9d99bff955089fa0d1ee5cc92ae167f78a...,83b55c7038f7f425a2f9838d9c3f1d06e469404f384f53...
2,2012-01,ANG MO KIO,314,df36be696d8b9d99bff955089fa0d1ee5cc92ae167f78a...,b403ffadae267afa1ef497228733c055092ed52da14963...


,month,town,block,street_name,storey_range,floor_area_sqm,Resale Identifier
1,2012-01,ANG MO KIO,314,ANG MO KIO AVE 3,07 TO 09,44.0,S3142501A
2,2012-01,ANG MO KIO,314,ANG MO KIO AVE 3,10 TO 12,44.0,S3142501A


## Checks
Validate row conservation, unchanged raw bytes, complete schema, date coverage,
lease arithmetic, reference adherence, output alignment and unique record hashes.
The separate pytest suite covers malformed values, deduplication ties, identifier
collisions, price outliers, missing inputs and a full synthetic offline rerun.


In [9]:
raw_after = {p.name: sha256_file(p) for p in sorted(INPUT_DIR.glob('*.csv'))}
assert raw_before == raw_after
assert counts['raw_rows'] == counts['out_of_scope_rows'] + counts['master_rows']
assert len(cleaned) + len(quarantined) == counts['master_rows']
assert set(cleaned._source_record_id).isdisjoint(quarantined._source_record_id)
assert len(cleaned) == len(transformed) == len(hashed)
assert set(summary['source_columns']).issubset(cleaned.columns)
assert cleaned.month.between(CONFIG.start_month, CONFIG.end_month).all()
assert (cleaned.remaining_lease_years * 12 + cleaned.remaining_lease_months).equals(cleaned.remaining_lease_total_months)
assert cleaned.remaining_lease_months.between(0, 11).all()
for attribute, allowed in reference['domains'].items():
    assert cleaned[attribute].isin(allowed).all()
assert hashed.resale_record_hash.is_unique
assert transformed['Resale Identifier'].str.fullmatch(r'S[0-9]{7}[A-Z]').all()
assert (OUTPUT_DIR / '_SUCCESS.json').exists()
print('All notebook checks passed. Raw files unchanged; every scoped row accounted for.')


All notebook checks passed. Raw files unchanged; every scoped row accounted for.


## Takeaways
- The strict January reference removes genuine later categories; do not use the
  cleaned data as an unbiased population sample. Version an approved reference
  expansion in production instead of changing labels silently.
- The specified duplicate key omits a transaction ID or unit number. It can merge
  real, separate sales. The maximum-price rule is followed exactly for valid rows,
  and all suppressed records remain available for audit.
- The short identifier has observed collisions. Use `resale_record_hash` for joins;
  the business code and its literal hash are descriptive fields.
- Lease expiry is estimated from coarse inputs. One supplied value differs from
  the estimate by 13 months and is explicitly marked for review.
- AWS deployment is an architecture deliverable only. Private Athena connectivity,
  IAM, driver compatibility and performance must be validated in the target account.

## Next steps
Read `README.md`, `docs/assumptions_and_insights.md` and `docs/architecture.md`.
The PNG diagrams and their SVG sources are included in `architecture/`.
